# iLQRの一反復の全体像

ここでは、具体的なそれぞれの導出は行わず、iLQRの一回の反復で何を行っているのかを理解する。反復といっているのは、制御では制御ループを繰り返すが、この繰り返しを反復と呼んでいる。

## 1. 添字の区別

iLQRは以下の図のように、実時間中のある時刻より伸ばした予測ホライゾン内でモデルのダイナミクスを考慮した制御入力列を計算する。

<p align ="center">
<img src="images/ilqr_moving_horizotal.png" style="width:45%;" />
</p>

iLQRには2種類の計算プロセスがあり、その区別を添字$k,i$で行う。

- $k = 0, \cdots , N$ : 上の図のように、予測ホライゾン内の時刻
- $i = 0, 1, \cdots $ : 入力列を改善する最適化反復の反復数

例えば、以下は「iLQRの第$i$反復で使っている、予測ホライゾン内の時刻$k$における基準入力」を表す。上付きのバーは基準であることを示し、後のステップで登場する線形化したモデルによるゲインを用いた制御入力により決定される元の非線形モデルの状態との差を表す基準であることを意味している。

$$
\bar{u}_k^{(i)}
$$

入力列全体は$\bar{U}^{(i)}$で表す。

$$
\bar{U}^{(i)} = [\bar{u}_0^{(i)} , \cdots , \bar{u}_{N-1}^{(i)}]
$$

iLQRはこの入力列を一回の制御ループ内で繰り返しによって以下のように改善していく方法である。

$$
\bar{U}^{(0)} \rightarrow \bar{U}^{(1)} \rightarrow \bar{U}^{(2)} \rightarrow \cdots
$$

## 2. 最初の基準軌道を作る

非線形状態方程式を離散時間で以下と表すとする。

$$
x_{k+1} = f(x_k , u_k)
$$

初期入力列 $\bar{U}^{(0)}$ を用意する。例えば、 $\bar{u}_k^{(0)}=0$としても良い。

この $\bar{U}^{(0)}$ を用いて、非線形状態方程式を初期状態 $x_0$ から順方向に計算を行う。初期状態の $x_0$ は予測ホライゾンの計算を始めるときの実システムの状態である。これを $\bar{x}_0^{(0)} = x_0$ としている。

$$
\bar{x}_{k+1}^{(0)} = f(\bar{x}_k^{(0)} , \bar{u}_k^{(0)}), \quad k = 0 , \cdots, N-1
$$

これより、最適化反復 0 回目の基準状態軌道が得られる。

$$
\bar{X}^{(0)} = [\bar{x}_0^{(0)}, \cdots, \bar{x}_0^{(0)}]
$$

この処理はrolloutである。

ここで、現在の状態と入力を合わせた基準軌道が完成する。

$$
\left( \bar{X}^{(0)}, \bar{U}^{(0)}\right)
$$

## 3. 基準軌道の周辺だけを近似する

基準軌道$\bar{x}_k^{(i)}$の近くに、別の状態$x_k$があるとする。これを以下のようにズレ$\delta x_k$を用いて表す。

$$
x_k = \bar{x}_k^{(i)} + \delta x_k
$$

よって、ズレ量は以下のようになる。

$$
\delta x_k = x_k - \bar{x}_k^{(i)}
$$

入力についても同様に考え、以下となる。

$$
\begin{aligned}
u_k = \bar{u}_k^{(i)} + \delta u_k \\
\delta u_k = u_k - \bar{u}_k^{(i)}
\end{aligned}
$$

これを非線形状態方程式へ代入する。

$$
x_{k+1} = f(\bar{x}_k^{(i)} + \delta x_k, \bar{u}_k^{(i)} + \delta u_k)
$$

基準点の周辺でTaylor展開すると次のようになる。

$$
x_{k+1} \approx f(\bar{x}_k^{(i)}, \bar{u}_k^{(i)}) + A_k \delta x_k + B_k \delta u_k
$$

$$
A_k = \left. \frac{\partial f}{\partial x} \right|_{\bar{x}_k^{(i)}, \bar{u}_k^{(i)}} , \quad B_k = \left. \frac{\partial f}{\partial u} \right|_{\bar{x}_k^{(i)}, \bar{u}_k^{(i)}} 
$$
 
 
基準軌道は以下であるため、

$$
\bar{x}_{k+1}^{(i)} = f(\bar{x}_k^{(i)}, \bar{u}_k^{(i)})
$$

これをTaylor展開の式に代入すると以下となる。

$$
x_{k+1} \approx \bar{x}_{k+1}^{(i)} + A_k \delta x_k + B_k \delta u_k
$$

よって、

$$
\delta x_{k+1} = x_{k+1} - \bar{x}_{k+1}^{(i)}
$$

と置けば、

$$
\delta x_{k+1} \approx A_k \delta x_k + B_k \delta u_k
$$

基準点周辺で非線形動力学の一次近似を表現できる。

コストについて、ステージコストと終端コストを基準軌道の周辺で二次近似する。

これにより、非線形最適制御問題を、基準軌道の周辺における局所的なLQR問題として捉える。具体的な内容はStep4で扱う。





## 4. backward passで改善則を構成する

局所近似した問題に対して、終端 $k=N$から時刻0へ逆向きに計算する。

その結果、各時刻における入力修正則を構成する。

$$
\delta u_k = d_k + K_l \delta x_k
$$

Step2と同様に、backward pass では具体的な入力列が求まるのではなく、以下の入力修正則を構成する係数列が求まる。

$$
d_0, \cdots, d_{N-1} \\
K_0, \cdots, K_{N-1}
$$

#### $d_k$ の役割

$d_k$ は基準入力自体をどちらに変更すればコストが下がるのかを表す。

例えば基準状態から$\delta x_k = 0$ とずれていなくても、以下の値となる。

$$
\delta u_k = d_k
$$

#### $K_k$の役割

$K_k \delta x_k$ はforward pass で生成される新しい状態が基準状態からずれたときの補正である。



## 5. forward pass で候補軌道を作る

初期状態は固定されているので、以下とする。

$$
x_0^{cand} = \bar{x}_0^{(i)} = x_0
$$

各時刻の候補入力を以下のように構成する。

$$
u_k^{cand} =\bar{u}_k^{(i)} + \alpha + K_k(x_k^{cand} - \bar{x}_k^{(i)})
$$

その入力をもとの非線形モデルへ与え、$x_{k+1}^{cand}$を求める。

$$
x_{k+1}^{cand} = f(x_k^{cand}, u_k^{cand})
$$

$$
u_0^{cand}(x_0^{cand}) \rightarrow x_1^{cand} = f(x_0^{cand}, u_0^{cand}) \rightarrow u_1^{cand}(x_1^{cand}) \rightarrow x_2^{cand} = f(x_1^{cand}, u_1^{cand}) \rightarrow \cdots
$$

以上のような関係であるため、$k=0$から$N-1$まで繰り返すことで、候補軌道が得られる。

$$
U^{cand}, X^{cand}
$$

この一連の順方向計算をforward pass と呼ぶ。